# KV-cache inference from scratch
A NumPy mini decoder-only transformer: generate with and without a KV cache, check the outputs match, then measure speed and memory.

## 1. Build the model (random weights, seed 42)

In [ ]:
import sys, os, time
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
from model import Config, init_params, n_params, forward_full, forward_cached, KVCache
from generate import generate_no_cache, generate_with_cache, compare
from memory import kv_bytes, memory_table
cfg = Config(); params = init_params(cfg, 42)
print(cfg, '| params:', n_params(params))
prompt = [int(x) for x in np.random.default_rng(42).integers(0, cfg.vocab_size, 8)]
print('prompt', prompt)

## 2. Prefill + one decode step equals a full forward pass
The cached path only feeds the *new* token, but reads K/V for every earlier position from the cache.

In [ ]:
full = forward_full(prompt, params, cfg)
cache = KVCache(cfg, 32)
pre = forward_cached(prompt[:5], params, cfg, cache)
rest = np.concatenate([forward_cached([t], params, cfg, cache) for t in prompt[5:]])
print('max abs diff:', np.abs(full - np.concatenate([pre, rest])).max(), '| cache length', cache.length)

## 3. Greedy generation: identical tokens, far less work

In [ ]:
r = compare(prompt, 56, params, cfg)
for k in ['tokens_identical', 'max_abs_logit_diff', 'tokens_processed_no_cache', 'tokens_processed_cache', 'time_no_cache_s', 'time_cache_s']:
    print(f'{k:28s}', r[k])
print('speedup %.1fx' % (r['time_no_cache_s'] / r['time_cache_s']))
print('generated:', r['generated'][:16], '...')

## 4. Speedup vs sequence length
No-cache work grows ~T^2/2 tokens; cached work grows ~T.

In [ ]:
for T in [16, 32, 64, 128]:
    r = compare(prompt, T - len(prompt), params, cfg)
    print(f"T={T:4d} speedup={r['time_no_cache_s']/r['time_cache_s']:5.1f}x  work ratio={r['tokens_processed_no_cache']/r['tokens_processed_cache']:5.1f}")

## 5. Memory: the price of the cache
KV bytes = 2 x layers x heads x T x d_head x bytes. Linear in T and in batch size.

In [ ]:
for row in memory_table(cfg, [128, 512, 1024], n_params(params) * 4):
    print(row)
print('GPT-2-small shape, fp16, T=1024: %.1f MiB' % (kv_bytes(12, 12, 64, 1024, 1, 2) / 2**20))

## 6. Full smoke run
`python run_smoke.py` rewrites `results/` (RESULTS.md, metrics.json, JSON.shot, SVG plots).